# Pembagian Data, Praproses Teks, dan Segmen Latih
Subbab 3.6.2 dan 3.6.3.

```
[1] Pembagian data        -> kolom split di manifest_rekaman.csv
[2] Kamus normalisasi     -> data/processed/kamus_normalisasi.json
[3] Uji aturan            -> data/output/praproses_teks/contoh_normalisasi.csv
[4] Normalisasi acuan     -> data/output/praproses_teks/acuan/<ID>.json
[5] Segmen latih          -> data/output/segmen_latih.csv, data/processed/segmen_latih/
```

## 0. Persiapan

In [18]:
# !pip install transformers==4.45.2 soundfile pandas

In [19]:
from pathlib import Path
import sys, json, datetime, importlib, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.praproses_teks as pt
importlib.reload(pt)

OUT        = ROOT / "data" / "output"
MASTER     = OUT / "master"
MANIFEST   = OUT / "manifest_rekaman.csv"
GLOSARIUM  = ROOT / "data" / "processed" / "glosarium" / "glosarium_semua.csv"
KAMUS_PATH = ROOT / "data" / "processed" / "kamus_normalisasi.json"
PRA_DIR    = OUT / "praproses_teks"
ACUAN_DIR  = PRA_DIR / "acuan"
AUDIO_PROC = ROOT / "data" / "processed" / "audio"
SEG_AUDIO  = ROOT / "data" / "processed" / "segmen_latih"

for d in (PRA_DIR, ACUAN_DIR):
    d.mkdir(parents=True, exist_ok=True)

rel = lambda p: p.relative_to(ROOT).as_posix()
print("Manifest  :", rel(MANIFEST), "| ada" if MANIFEST.exists() else "| TIDAK ADA")
print("Master    :", rel(MASTER), f"| {len(list(MASTER.glob('*.tsv')))} berkas")
print("Glosarium :", rel(GLOSARIUM), "| ada" if GLOSARIUM.exists() else "| TIDAK ADA")
print("Audio     :", rel(AUDIO_PROC), "| ada" if AUDIO_PROC.exists() else "| tidak ada (pemotongan audio dilewati)")

Manifest  : data/output/manifest_rekaman.csv | ada
Master    : data/output/master | 35 berkas
Glosarium : data/processed/glosarium/glosarium_semua.csv | ada
Audio     : data/processed/audio | ada


In [20]:
manifest = pd.read_csv(MANIFEST, dtype=str, keep_default_na=False, encoding="utf-8-sig")

master = pd.concat(
    [pd.read_csv(p, sep="\t", dtype=str, keep_default_na=False, encoding="utf-8-sig")
     for p in sorted(MASTER.glob("*.tsv"))],
    ignore_index=True)
for k in ("start", "end", "dur"):
    master[k] = master[k].astype(float)
master["seg_id"] = master["seg_id"].astype(int)
master = master.sort_values(["file_id", "start"]).reset_index(drop=True)

id_manifest, id_master = set(manifest.file_id), set(master.file_id)
assert id_manifest == id_master, (id_manifest ^ id_master)

durasi = manifest.set_index("file_id")["durasi_potong_detik"].astype(float)
kata = master.groupby("file_id")["teks_acuan"].apply(lambda s: s.str.split().str.len().sum())
print(f"{len(manifest)} rekaman, {len(master)} segmen, {durasi.sum()/3600:.2f} jam, {kata.sum()} kata")

35 rekaman, 6911 segmen, 9.78 jam, 63480 kata


## 1. Pembagian data (3.6.2)
Daftar ditetapkan sebagai konstanta, lalu diperiksa terhadap ketentuan 3.6.2: perbandingan durasi 65 : 10 : 25 (toleransi 3 poin), setiap mata kuliah ada di uji, dan setiap rekaman masuk tepat satu himpunan. Kolom `split` di manifest diisi otomatis.

In [21]:
SPLIT = {
    "uji": ["AOK-04", "AOK-07", "APS-03", "APS-05", "IMK-02",
            "JARNIR-03", "PKK-02", "RPL-07", "RPL-10"],
    "validasi": ["AOK-01", "APS-02", "JARNIR-05", "RPL-02"],
    "latih": ["AOK-02", "AOK-03", "AOK-05", "AOK-06", "AOK-08",
              "APS-01", "APS-04", "APS-06",
              "IMK-01", "IMK-03", "IMK-04", "IMK-05",
              "PKK-01", "PKK-03", "PKK-05",
              "RPL-03", "RPL-04", "RPL-06", "RPL-09", "RPL-11",
              "JARNIR-01", "JARNIR-04"],
}
TARGET = {"latih": 65.0, "validasi": 10.0, "uji": 25.0}
TOLERANSI_POIN = 3.0
URUT = ["latih", "validasi", "uji"]

split_of = {fid: s for s, ids in SPLIT.items() for fid in ids}

In [22]:
masalah = []

# Ketentuan 3: tingkat rekaman
semua = [fid for ids in SPLIT.values() for fid in ids]
ganda = sorted({f for f in semua if semua.count(f) > 1})
if ganda:
    masalah.append(f"rekaman di lebih dari satu himpunan: {ganda}")
if set(semua) != id_manifest:
    masalah.append(f"tidak cocok dengan manifest: kurang {sorted(id_manifest - set(semua))}, "
                   f"lebih {sorted(set(semua) - id_manifest)}")
url_ganda = manifest[manifest.url.duplicated(keep=False)]
if len(url_ganda):
    masalah.append(f"video yang sama muncul dua kali: {url_ganda.file_id.tolist()}")

m = manifest.assign(split=manifest.file_id.map(split_of),
                    durasi=manifest.durasi_potong_detik.astype(float),
                    kata=manifest.file_id.map(kata))

# Ketentuan 2: stratifikasi
tanpa_uji = sorted(set(m.mata_kuliah) - set(m[m.split == "uji"].mata_kuliah))
if tanpa_uji:
    masalah.append(f"mata kuliah tanpa rekaman uji: {tanpa_uji}")

# Ketentuan 1: perbandingan
ring = (m.groupby("split").agg(rekaman=("file_id", "count"), detik=("durasi", "sum"), kata=("kata", "sum"))
          .reindex(URUT))
ring["%_rekaman"] = (100 * ring.rekaman / ring.rekaman.sum()).round(1)
ring["%_durasi"] = (100 * ring.detik / ring.detik.sum()).round(1)
ring["%_kata"] = (100 * ring.kata / ring.kata.sum()).round(1)
ring["target_%"] = pd.Series(TARGET)
ring["jam"] = (ring.detik / 3600).round(2)
for s in URUT:
    if abs(ring.at[s, "%_durasi"] - TARGET[s]) > TOLERANSI_POIN:
        masalah.append(f"porsi durasi {s} {ring.at[s, '%_durasi']}% menyimpang dari {TARGET[s]}%")

print(ring[["rekaman", "jam", "kata", "%_rekaman", "%_durasi", "%_kata", "target_%"]].to_string())
assert not masalah, "\n".join(masalah)
print("\nKetentuan 1 sampai 3 terpenuhi.")

          rekaman   jam   kata  %_rekaman  %_durasi  %_kata  target_%
split                                                                
latih          22  6.36  40823       62.9      65.1    64.3      65.0
validasi        4  0.98   7133       11.4      10.0    11.2      10.0
uji             9  2.44  15524       25.7      24.9    24.5      25.0

Ketentuan 1 sampai 3 terpenuhi.


In [23]:
per_mk = (m.pivot_table(index="mata_kuliah", columns="split", values="durasi",
                        aggfunc=["count", "sum"], fill_value=0))
tabel_mk = pd.DataFrame({
    "rekaman L/V/U": [f"{per_mk[('count', 'latih')][mk]}/{per_mk[('count', 'validasi')][mk]}/{per_mk[('count', 'uji')][mk]}"
                      for mk in per_mk.index],
    "menit latih": (per_mk[("sum", "latih")] / 60).round(1),
    "menit validasi": (per_mk[("sum", "validasi")] / 60).round(1),
    "menit uji": (per_mk[("sum", "uji")] / 60).round(1),
}, index=per_mk.index)
tabel_mk["% durasi uji"] = (100 * per_mk[("sum", "uji")] / per_mk["sum"].sum(axis=1)).round(1)
print(tabel_mk.to_string())

tanpa_val = sorted(set(m.mata_kuliah) - set(m[m.split == "validasi"].mata_kuliah))
print("\nMata kuliah tanpa rekaman validasi:", ", ".join(tanpa_val) or "-")
print("Ini tidak melanggar ketentuan 2: validasi hanya dipakai memilih checkpoint berdasarkan WER.")

# Ketentuan 4: penutur, hanya dilaporkan
p = m[m.penutur != ""]
if len(p):
    print("\nPenutur yang tercatat:")
    print(p.pivot_table(index=["mata_kuliah", "penutur"], columns="split", values="file_id",
                        aggfunc=lambda x: " ".join(sorted(x)), fill_value="").to_string())

# Blok tiga menit pada himpunan uji (satuan analisis subbab 3.4)
u = m[m.split == "uji"].set_index("file_id")
blok = np.ceil(u.durasi / 180).astype(int)
print(f"\nBlok tiga menit pada himpunan uji: {blok.sum()} blok, termasuk blok sisa di akhir tiap rekaman "
      f"({', '.join(f'{k} {v}' for k, v in blok.items())})")

            rekaman L/V/U  menit latih  menit validasi  menit uji  % durasi uji
mata_kuliah                                                                    
AOK                 5/1/2         92.9            13.9       35.1          24.7
APS                 3/1/2         45.9            11.1       20.4          26.4
IMK                 4/0/1         65.6             0.0       18.7          22.2
JARNIR              2/1/1         38.0            14.7       18.2          25.6
PKK                 3/0/1         49.5             0.0       16.9          25.4
RPL                 5/1/2         89.8            19.0       37.0          25.4

Mata kuliah tanpa rekaman validasi: IMK, PKK
Ini tidak melanggar ketentuan 2: validasi hanya dipakai memilih checkpoint berdasarkan WER.

Penutur yang tercatat:
split                        latih            uji validasi
mata_kuliah penutur                                       
APS         P1              APS-01  APS-03 APS-05         
            P2       

In [24]:
lama = manifest["split"].where(manifest["split"] != "", None)
baru = manifest["file_id"].map(split_of)
beda = manifest[lama.notna() & (lama != baru)]
TIMPA_SPLIT_LAMA = False
assert beda.empty or TIMPA_SPLIT_LAMA, (
    "Kolom split di manifest sudah berisi nilai lain:\n"
    + beda[["file_id", "split"]].to_string(index=False)
    + "\nUbah TIMPA_SPLIT_LAMA menjadi True bila memang ingin menimpanya.")

manifest["split"] = baru
manifest.to_csv(MANIFEST, index=False, encoding="utf-8-sig")
ring.reset_index().to_csv(PRA_DIR / "ringkasan_split.csv", index=False, encoding="utf-8-sig")
tabel_mk.reset_index().to_csv(PRA_DIR / "ringkasan_split_per_mk.csv", index=False, encoding="utf-8-sig")
print("Kolom split ditulis ke", rel(MANIFEST))
print(manifest.split.value_counts().reindex(URUT).to_string())

Kolom split ditulis ke data/output/manifest_rekaman.csv
split
latih       22
validasi     4
uji          9


## 2. Kamus normalisasi (3.6.3)
Kandidat dicari dari transkrip latih dan validasi saja. Kamus disimpan sebagai draf; setelah ditinjau, ubah `status` menjadi `"terkunci"` dan isi `tanggal_kunci`, lalu jalankan ulang notebook.

In [25]:
import re, collections
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
_basic = BasicTextNormalizer()

pindai = master[master.file_id.map(split_of).isin(["latih", "validasi"])]
frek = collections.Counter(w for s in pindai.teks_acuan for w in _basic(s).split())

def kandidat(pola):
    return {w: c for w, c in frek.most_common() if re.fullmatch(pola, w)}

print("Dipindai:", pindai.file_id.nunique(), "rekaman latih dan validasi,", sum(frek.values()), "kata\n")
print("Jeda terisi       :", kandidat(r"e+|e+h+|e+m+|h+m+|m{2,}|a{2,}h*|u+m+|u+h+|e+h+m+"))
print("Negasi lisan      :", kandidat(r"n?g+a+k?|e?n?g+a+k?|gk|nda+k?"))
print("Singkatan tulis   :", kandidat(r"yg|dgn|dg|tdk|utk|krn|sdh|spt|dll|dsb|dst|tsb|pd|dr|dlm|bs|jd"))

Dipindai: 26 rekaman latih dan validasi, 48272 kata

Jeda terisi       : {'e': 182, 'eee': 39, 'ee': 2, 'eem': 1, 'aa': 1}
Negasi lisan      : {'gak': 6, 'enggak': 5, 'engga': 3, 'nggak': 2, 'ga': 1}
Singkatan tulis   : {}


In [26]:
if not KAMUS_PATH.exists():
    pt.simpan_kamus(pt.KAMUS_DRAF, KAMUS_PATH)
    print("Draf kamus dibuat:", rel(KAMUS_PATH))
else:
    print("Kamus sudah ada, tidak ditimpa:", rel(KAMUS_PATH))

kamus = pt.muat_kamus(KAMUS_PATH)

baris = ([("ejaan_lisan", k, v) for k, v in kamus.ejaan_lisan.items()]
         + [("singkatan", k, v) for k, v in kamus.singkatan.items()]
         + [("jeda_terisi", k, "(dibuang)") for k in sorted(kamus.jeda_terisi)])
tabel_kamus = pd.DataFrame(baris, columns=["kelompok", "bentuk", "menjadi"])
tabel_kamus["muncul di latih+validasi"] = tabel_kamus.bentuk.map(frek).fillna(0).astype(int)
print(tabel_kamus.to_string(index=False))
tabel_kamus.to_csv(PRA_DIR / "kamus_normalisasi.csv", index=False, encoding="utf-8-sig")

if kamus.terkunci:
    print(f"\nKamus terkunci sejak {kamus.tanggal_kunci}.")
else:
    print("\n[!] Kamus masih berstatus DRAF. Tinjau isinya, lalu kunci sebelum Tahap 1 dijalankan.")

Kamus sudah ada, tidak ditimpa: data/processed/kamus_normalisasi.json
   kelompok bentuk        menjadi  muncul di latih+validasi
ejaan_lisan    gak         enggak                         6
ejaan_lisan     ga         enggak                         1
ejaan_lisan  engga         enggak                         3
ejaan_lisan   ngga         enggak                         0
ejaan_lisan  nggak         enggak                         2
  singkatan     yg           yang                         0
  singkatan    dgn         dengan                         0
  singkatan    tdk          tidak                         0
  singkatan    utk          untuk                         0
  singkatan    dll  dan lain lain                         0
  singkatan    dsb dan sebagainya                         0
  singkatan    dst dan seterusnya                         0
  singkatan    tsb       tersebut                         0
jeda_terisi      e      (dibuang)                       182
jeda_terisi     ee      (dibua

## 3. Uji aturan normalisasi
Semua uji harus lolos sebelum lanjut. Tabelnya disimpan untuk lampiran.

In [27]:
glosarium = pt.muat_glosarium(GLOSARIUM)

UJI_ATURAN = [
    # (kelompok, mata kuliah untuk glosarium atau None, masukan, harapan)
    ("Tabel 3.4: bentuk ulang", None, "Kata-kata itu", "kata kata itu"),
    ("Tabel 3.4: gabungan kata", None, "media pandang-dengar", "media pandang dengar"),
    ("Tabel 3.4: unsur asing", None, "datanya di-encode dulu", "datanya di encode dulu"),
    ("Tabel 3.4: hubung menggantung", None, "komu- komunikasi", "komu komunikasi"),
    ("Tabel 3.4: dua tanda hubung", None, "jaringan--nirkabel", "jaringan nirkabel"),
    ("Tabel 3.4: istilah bertanda hubung", "APS", "sistem real-time", "sistem real time"),
    ("Dasar: huruf kecil dan tanda baca", None, "Oke, jadi CPU-nya sibuk?", "oke jadi cpu nya sibuk"),
    ("Dasar: teks dalam kurung", None, "lalu (musik) masuk", "lalu masuk"),
    ("Bilangan: majemuk", None, "dua puluh lima", "25"),
    ("Bilangan: tahun", None, "tahun seribu sembilan ratus empat puluh", "tahun 1940"),
    ("Bilangan: angka dan pengali", None, "sekitar 2 ribu", "sekitar 2000"),
    ("Bilangan: pemisah ribuan", None, "sekitar 2.000", "sekitar 2000"),
    ("Bilangan: titik di antara angka", None, "delapan ratus dua titik sebelas", "802 11"),
    ("Bilangan: angka bertitik", None, "802.11", "802 11"),
    ("Bilangan: koma desimal", None, "satu koma empat puluh empat", "1 44"),
    ("Bilangan: persen", None, "10%", "10 persen"),
    ("Bilangan: deret satuan", None, "satu dua tiga", "1 2 3"),
    ("Bilangan: pengali tanpa satuan", None, "beberapa ratus orang", "beberapa ratus orang"),
    ("Bilangan: kata lain tetap", None, "salah satu", "salah 1"),
    ("Akronim: huruf terpisah", None, "u m l", "uml"),
    ("Akronim: garis miring", None, "I/O", "io"),
    ("Akronim: tetap rapat", None, "UML", "uml"),
    ("Kamus: varian ejaan", None, "gak, nggak, ngga, engga", "enggak enggak enggak enggak"),
    ("Kamus: bentuk lisan lain dibiarkan", None, "udah gitu aja", "udah gitu aja"),
    ("Kamus: singkatan tulis", None, "yg dgn tdk", "yang dengan tidak"),
    ("Kamus: jeda terisi", None, "eee jadi e ini emm", "jadi ini"),
    ("Glosarium: dilindungi dari kamus", "PKK", "platform e-learning", "platform e learning"),
    ("Glosarium: di luar istilah tetap kena", "PKK", "e platform", "platform"),
]

hasil = []
for kel, mk, masukan, harap in UJI_ATURAN:
    norm = pt.Normalisasi(kamus, glosarium.get(mk) if mk else None)
    dapat = norm.teks(masukan)
    hasil.append({"kelompok": kel, "masukan": masukan, "hasil": dapat, "harapan": harap,
                  "lolos": dapat == harap})
uji = pd.DataFrame(hasil)
print(uji[["kelompok", "masukan", "hasil", "lolos"]].to_string(index=False))
assert uji.lolos.all(), uji[~uji.lolos].to_string(index=False)
uji.drop(columns="lolos").to_csv(PRA_DIR / "contoh_normalisasi.csv", index=False, encoding="utf-8-sig")
print(f"\nSeluruh {len(uji)} uji lolos.")

                             kelompok                                 masukan                       hasil  lolos
              Tabel 3.4: bentuk ulang                           Kata-kata itu               kata kata itu   True
             Tabel 3.4: gabungan kata                    media pandang-dengar        media pandang dengar   True
               Tabel 3.4: unsur asing                  datanya di-encode dulu      datanya di encode dulu   True
        Tabel 3.4: hubung menggantung                        komu- komunikasi             komu komunikasi   True
          Tabel 3.4: dua tanda hubung                      jaringan--nirkabel           jaringan nirkabel   True
   Tabel 3.4: istilah bertanda hubung                        sistem real-time            sistem real time   True
    Dasar: huruf kecil dan tanda baca                Oke, jadi CPU-nya sibuk?      oke jadi cpu nya sibuk   True
             Dasar: teks dalam kurung                      lalu (musik) masuk                  l

## 4. Normalisasi acuan dan penandaan glosarium
Tiap segmen dinormalisasi tersendiri agar token tetap terikat pada stempel waktunya; penandaan istilah dijalankan atas satu rekaman utuh. Berkas `acuan/<ID>.json` berisi `token`, `segmen`, `istilah`, dan `G`, dan menjadi masukan Tahap 1.

In [28]:
baris_tsv, ringkas = [], []
for fid, g in master.groupby("file_id", sort=True):
    mk = g.mata_kuliah.iloc[0]
    norm = pt.Normalisasi(kamus, glosarium[mk])
    token, segmen = [], []
    for r in g.itertuples():
        t = norm(r.teks_acuan)
        segmen.append({"seg_id": int(r.seg_id), "start": round(r.start, 3), "end": round(r.end, 3),
                       "i0": len(token), "i1": len(token) + len(t)})
        token.extend(t)
    istilah = pt.tandai_glosarium(token, glosarium[mk])
    G = pt.himpunan_g(istilah)
    dok = {"file_id": fid, "mata_kuliah": mk, "split": split_of[fid],
           "kamus": {"status": kamus.status, "tanggal_kunci": kamus.tanggal_kunci},
           "token": token, "segmen": segmen,
           "istilah": [{"i0": a, "i1": b, "entri_id": e} for a, b, e in istilah],
           "G": G}
    (ACUAN_DIR / f"{fid}.json").write_text(json.dumps(dok, ensure_ascii=False), encoding="utf-8")

    Gs = set(G)
    for s, r in zip(segmen, g.itertuples()):
        baris_tsv.append({"file_id": fid, "mata_kuliah": mk, "split": split_of[fid], "seg_id": s["seg_id"],
                          "start": s["start"], "end": s["end"], "teks_acuan": r.teks_acuan,
                          "teks_normal": " ".join(token[s["i0"]:s["i1"]]),
                          "n_token": s["i1"] - s["i0"],
                          "n_G": sum(1 for k in range(s["i0"], s["i1"]) if k in Gs)})
    ringkas.append({"file_id": fid, "mata_kuliah": mk, "split": split_of[fid],
                    "kata_asli": int(kata[fid]), "token": len(token), "G": len(G),
                    "kemunculan_istilah": len(istilah), "entri_berbeda": len({e for *_, e in istilah})})

acuan = pd.DataFrame(baris_tsv)
acuan.to_csv(PRA_DIR / "acuan_ternormalisasi.tsv", sep="\t", index=False, encoding="utf-8-sig")
ring_acuan = pd.DataFrame(ringkas)
ring_acuan["%G"] = (100 * ring_acuan.G / ring_acuan.token).round(1)
ring_acuan.to_csv(PRA_DIR / "ringkasan_acuan.csv", index=False, encoding="utf-8-sig")
print(f"{len(ring_acuan)} rekaman ditulis ke {rel(ACUAN_DIR)}")
print(f"Token: {ring_acuan.kata_asli.sum()} kata asli -> {ring_acuan.token.sum()} token ternormalisasi")

35 rekaman ditulis ke data/output/praproses_teks/acuan
Token: 63480 kata asli -> 63483 token ternormalisasi


In [29]:
def agregat(by):
    a = ring_acuan.groupby(by)[["token", "G", "kemunculan_istilah"]].sum()
    a["%G"] = (100 * a.G / a.token).round(1)
    return a

print("Per himpunan:")
print(agregat("split").reindex(URUT).to_string())
print("\nPer mata kuliah pada himpunan uji:")
uji_mk = ring_acuan[ring_acuan.split == "uji"].groupby("mata_kuliah")[["token", "G", "kemunculan_istilah"]].sum()
uji_mk["%G"] = (100 * uji_mk.G / uji_mk.token).round(1)
print(uji_mk.to_string())
print("\nRekaman uji:")
print(ring_acuan[ring_acuan.split == "uji"].drop(columns="split").to_string(index=False))

Per himpunan:
          token     G  kemunculan_istilah    %G
split                                          
latih     40733  3900                3222   9.6
validasi   7144   721                 616  10.1
uji       15606  1406                1149   9.0

Per mata kuliah pada himpunan uji:
             token    G  kemunculan_istilah    %G
mata_kuliah                                      
AOK           3214  454                 364  14.1
APS           2197  316                 238  14.4
IMK           2311  107                  84   4.6
JARNIR        1751  108                  99   6.2
PKK           1150  110                  94   9.6
RPL           4983  311                 270   6.2

Rekaman uji:
  file_id mata_kuliah  kata_asli  token   G  kemunculan_istilah  entri_berbeda   %G
   AOK-04         AOK       1736   1759 286                 234             29 16.3
   AOK-07         AOK       1432   1455 168                 130             20 11.5
   APS-03         APS       1138   1158 125 

In [30]:
# Istilah yang paling sering muncul per mata kuliah, untuk pembahasan TTER pada Bab 4.
gl = pd.read_csv(GLOSARIUM, dtype=str, keep_default_na=False)
kanonik = gl.drop_duplicates("entri_id").set_index("entri_id")["bentuk_kanonik"]

hit = []
for p in sorted(ACUAN_DIR.glob("*.json")):
    d = json.loads(p.read_text(encoding="utf-8"))
    hit += [{"mata_kuliah": d["mata_kuliah"], "split": d["split"], "entri_id": x["entri_id"],
             "token": x["i1"] - x["i0"]} for x in d["istilah"]]
hit = pd.DataFrame(hit)
hit["bentuk_kanonik"] = hit.entri_id.map(kanonik)

entri_muncul = hit.groupby("mata_kuliah").entri_id.nunique()
print("Entri glosarium yang muncul di transkrip acuan (dari 150 per mata kuliah):")
print(entri_muncul.to_string())

print("\nSepuluh entri tersering per mata kuliah dan porsinya terhadap seluruh kemunculan:")
for mk, h in hit.groupby("mata_kuliah"):
    c = h.bentuk_kanonik.value_counts()
    porsi = 100 * c.head(10).sum() / c.sum()
    print(f"  {mk:6s} ({porsi:.0f}%): " + ", ".join(f"{k} ({v})" for k, v in c.head(10).items()))
hit.groupby(["mata_kuliah", "split", "entri_id", "bentuk_kanonik"]).size().rename("kemunculan") \
   .reset_index().to_csv(PRA_DIR / "kemunculan_istilah.csv", index=False, encoding="utf-8-sig")

Entri glosarium yang muncul di transkrip acuan (dari 150 per mata kuliah):
mata_kuliah
AOK       73
APS       58
IMK       55
JARNIR    31
PKK       48
RPL       63

Sepuluh entri tersering per mata kuliah dan porsinya terhadap seluruh kemunculan:
  AOK    (57%): memori (189), CPU (95), I/O (69), register (62), instruction (61), cache (56), prosesor (53), RAM (44), main memory (34), bus (34)
  APS    (70%): sistem (198), perancangan (160), analisis (60), kebutuhan (44), perangkat lunak (40), kelas (39), model (32), diagram (29), komponen (24), interface (22)
  IMK    (70%): user (206), evaluation (74), interaction design (55), interaction (44), requirements (44), design (43), goal (38), human-computer interaction (27), human (21), computer (21)
  JARNIR (83%): nirkabel (53), local area network (51), jaringan nirkabel (39), personal area network (20), sinyal (19), wide area network (18), Internet of Things (15), Wi-Fi (13), wireless sensor network (12), protokol (11)
  PKK    (67%): kom

In [31]:
# Contoh segmen sebelum dan sesudah normalisasi, untuk lampiran.
# Hanya segmen pendek yang hasilnya berbeda dari BasicTextNormalizer saja, supaya efek aturan tambahan terlihat.
ATURAN = [
    ("tanda hubung", r"\w-\w"),
    ("kata bilangan", r"(?i)\b(satu|dua|tiga|empat|lima|enam|tujuh|delapan|sembilan|sepuluh|sebelas|seratus|seribu|puluh|ratus|ribu|belas)\b"),
    ("angka bertanda baca", r"\d[.,%]"),
    ("garis miring", r"\w/\w"),
    ("jeda terisi", r"(?i)\b(e+|e+m+|h+m+)\b"),
    ("ejaan lisan", r"(?i)\b(gak|ga|nggak|ngga|engga)\b"),
]
def tanda(teks):
    return [n for n, pola in ATURAN if re.search(pola, teks)]

acuan["hanya_dasar"] = acuan.teks_acuan.map(lambda t: " ".join(_basic(t).split()))
kandidat_contoh = acuan[(acuan.teks_normal != acuan.hanya_dasar) | acuan.teks_acuan.str.contains(r"\w-\w")]
kandidat_contoh = kandidat_contoh[kandidat_contoh.teks_acuan.str.split().str.len() <= 14]

kandidat_contoh = kandidat_contoh.assign(aturan=kandidat_contoh.teks_acuan.map(lambda t: ", ".join(tanda(t))))
contoh = (kandidat_contoh[kandidat_contoh.aturan != ""].groupby("aturan").head(2).sort_values("aturan")
          [["file_id", "seg_id", "aturan", "teks_acuan", "teks_normal"]])
contoh.to_csv(PRA_DIR / "contoh_acuan.csv", index=False, encoding="utf-8-sig")
with pd.option_context("display.max_colwidth", 90):
    print(contoh.head(16).to_string(index=False))

file_id  seg_id                     aturan                                                              teks_acuan                                                                 teks_normal
 APS-06      63        angka bertanda baca Sebaiknya, luangkan waktu lebih sekitar 10% ketika membuat penjadwalan. sebaiknya luangkan waktu lebih sekitar 10 persen ketika membuat penjadwalan
 IMK-05      38        angka bertanda baca                            Apakah ingin 100% dicapai atau 80% atau 70%?               apakah ingin 100 persen dicapai atau 80 persen atau 70 persen
 AOK-02     394                ejaan lisan                        Sesuatu yang dibagi 0 kan gak bisa tak terhingga                         sesuatu yang dibagi 0 kan enggak bisa tak terhingga
 AOK-02     482                ejaan lisan                                                      Ada interrupt gak?                                                        ada interrupt enggak
 AOK-02      44               garis miring   

## 5. Segmen latih (3.6.2)
Hanya untuk rekaman latih dan validasi. Segmen acuan berurutan digabung sampai paling lama 30 detik, dan berhenti bila celahnya lebih dari 1 detik atau rekaman berakhir. Labelnya teks acuan asli, bukan hasil normalisasi.

In [32]:
MAKS_DURASI = 30.0
MAKS_CELAH = 1.0

baris = []
for fid, g in master[master.file_id.map(split_of).isin(["latih", "validasi"])].groupby("file_id", sort=True):
    gabung = pt.gabung_segmen(g.to_dict("records"), MAKS_DURASI, MAKS_CELAH)
    for k, h in enumerate(gabung, start=1):
        baris.append({"seg_latih_id": f"{fid}_{k:03d}", "file_id": fid, "mata_kuliah": g.mata_kuliah.iloc[0],
                      "split": split_of[fid], "start": round(h["start"], 3), "end": round(h["end"], 3),
                      "dur": round(h["dur"], 3), "seg_id_awal": h["seg_id_awal"], "seg_id_akhir": h["seg_id_akhir"],
                      "n_segmen_asal": h["n_segmen_asal"], "n_kata": len(h["teks"].split()), "teks": h["teks"]})
seg_latih = pd.DataFrame(baris)

# Pemeriksaan: tidak ada yang melewati 30 detik, dan setiap segmen asal terpakai tepat sekali.
assert (seg_latih.dur <= MAKS_DURASI + 1e-6).all()
asal = master[master.file_id.map(split_of).isin(["latih", "validasi"])]
assert seg_latih.n_segmen_asal.sum() == len(asal)

seg_latih.to_csv(OUT / "segmen_latih.csv", index=False, encoding="utf-8-sig")

r = seg_latih.groupby("split").agg(segmen=("seg_latih_id", "count"), jam=("dur", lambda x: x.sum() / 3600),
                                   median_detik=("dur", "median"), maks_kata=("n_kata", "max"))
r["%>=25 detik"] = seg_latih.groupby("split").dur.apply(lambda x: 100 * (x >= 25).mean())
r["%<10 detik"] = seg_latih.groupby("split").dur.apply(lambda x: 100 * (x < 10).mean())
print(r.round(2).reindex(["latih", "validasi"]).to_string())
print(f"\nDari {len(asal)} segmen asal menjadi {len(seg_latih)} segmen latih; ditulis ke {rel(OUT / 'segmen_latih.csv')}")

          segmen   jam  median_detik  maks_kata  %>=25 detik  %<10 detik
split                                                                   
latih       1001  5.99         24.89         83        49.65       14.19
validasi     149  0.95         23.32         71        45.64        2.68

Dari 5120 segmen asal menjadi 1150 segmen latih; ditulis ke data/output/segmen_latih.csv


In [33]:
# Pemotongan audio. Membutuhkan data/processed/audio/<ID>.wav hasil praproses audio (subbab 3.6.1).
POTONG_AUDIO = True
TIMPA_AUDIO = False

if not POTONG_AUDIO:
    print("POTONG_AUDIO False, dilewati.")
elif not AUDIO_PROC.exists():
    print(f"{rel(AUDIO_PROC)} tidak ditemukan, pemotongan dilewati.")
else:
    hilang = sorted(set(seg_latih.file_id) - {p.stem for p in AUDIO_PROC.glob('*.wav')})
    assert not hilang, f"WAV tidak ditemukan untuk: {hilang}"
    n_baru = 0
    for r in seg_latih.itertuples():
        tujuan = SEG_AUDIO / r.split / f"{r.seg_latih_id}.wav"
        if tujuan.exists() and not TIMPA_AUDIO:
            continue
        pt.potong_audio(AUDIO_PROC / f"{r.file_id}.wav", r.start, r.end, tujuan)
        n_baru += 1
    for s, d in seg_latih.groupby("split"):
        (d.assign(file_name=d.seg_latih_id + ".wav")
          .rename(columns={"teks": "transcription"})
          [["file_name", "transcription", "file_id", "start", "end", "dur"]]
          .to_csv(SEG_AUDIO / s / "metadata.csv", index=False, encoding="utf-8"))
    print(f"{n_baru} berkas baru dipotong ke {rel(SEG_AUDIO)}")

    import soundfile as sf
    contoh_wav = sorted((SEG_AUDIO / "latih").glob("*.wav"))[0]
    i = sf.info(str(contoh_wav))
    print(f"Contoh {contoh_wav.name}: {i.samplerate} Hz, {i.channels} kanal, {i.subtype}, {i.duration:.2f} detik")

0 berkas baru dipotong ke data/processed/segmen_latih
Contoh AOK-02_001.wav: 16000 Hz, 1 kanal, PCM_16, 29.21 detik


## 6. Metadata run

In [34]:
import platform, transformers
meta = {
    "tanggal": datetime.datetime.now().isoformat(timespec="seconds"),
    "tahap": "pembagian data, praproses teks, segmen latih",
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "split": SPLIT,
    "kamus": {"status": kamus.status, "tanggal_kunci": kamus.tanggal_kunci},
    "segmen_latih": {"maks_durasi": MAKS_DURASI, "maks_celah": MAKS_CELAH},
    "versi_pustaka": {"transformers": transformers.__version__, "pandas": pd.__version__, "numpy": np.__version__},
}
try:
    import soundfile
    meta["versi_pustaka"]["soundfile"] = soundfile.__version__
except ImportError:
    pass
(PRA_DIR / "run_meta_praproses.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(meta["versi_pustaka"], indent=2))

{
  "transformers": "4.45.2",
  "pandas": "3.0.3",
  "numpy": "2.0.2",
  "soundfile": "0.13.1"
}


## Output

| Himpunan | Audio | Acuan / label | Dipakai di |
|---|---|---|---|
| Latih (22 rekaman) | `data/processed/segmen_latih/latih/*.wav` (potongan ≤30 detik) | `metadata.csv` di folder yang sama | Tahap 3: melatih LoRA |
| Validasi (4 rekaman) | `data/processed/segmen_latih/validasi/*.wav` (potongan ≤30 detik) | `metadata.csv` di folder yang sama | Tahap 3: memilih checkpoint (WER) |
| Uji (9 rekaman) | `data/processed/audio/<ID>.wav` (utuh, tidak dipotong) | `data/output/praproses_teks/acuan/<ID>.json` | Tahap 1–3: mengukur WER, CER, TTER, RTF |

Berkas lain:
- `data/processed/kamus_normalisasi.json`: kamus penggantian, dipakai Tahap 1–3 untuk menormalisasi hipotesis
- `data/output/praproses_teks/`: tabel ringkasan dan contoh normalisasi untuk Bab 4 dan lampiran